In [6]:
import numpy as np
import matplotlib.pyplot as plt
import gzip

import os
import pickle

import matplotlib.gridspec as gridspec

In [7]:
###########################################################

notebook_dir = os.getcwd()

os.chdir(notebook_dir)

###########################################################

In [8]:
# Load Q Sim data
try:
    with open('../QSim_Data/Fig_4_QSim_data.pkl', 'rb') as f:
        mag_data, kink_data = pickle.load(f)

except:
    print("Something went wrong")

kink_slice = {}
mag_slice = {}
x_local = {}
bond_x_local = {}
tauQ_list_qsim = sorted(list(mag_data.keys()))
for tauQ in tauQ_list_qsim:
    datamag = mag_data[tauQ]
    datakink = kink_data[tauQ]

    bond_x_local[tauQ] = datakink[0]
    kink_slice[tauQ] = datakink[1]
    x_local[tauQ] = datamag[0]
    mag_slice[tauQ] = datamag[1]

In [9]:
# load disordered MPS data
try:
    # Unpack the tuple back into your two variables
    with open('Fig_4_disordered_MPS_data.pkl', 'rb') as f:
        averaged_data = pickle.load(f)

except:
    print("Something went wrong")

final_Mz_spatial = {}
final_Ezz_spatial = {}

valid_ta_list = sorted(list(averaged_data.keys()))

for ta in valid_ta_list:
    data = averaged_data[ta]
    Ez_avg = data['Ez']
    Ezz_avg = data['Ezz']
    
    # Grab the VERY LAST time step using index [-1]
    Ez_final = Ez_avg[-1, :]            
    Ezz_final = Ezz_avg[-1, :, :] if Ezz_avg.ndim == 3 else Ezz_avg[-1, :]     
    
    # Store spatial data for plotting
    final_Mz_spatial[ta] = Ez_final
    final_Ezz_spatial[ta] = Ezz_final

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ==========================================
# 1. Global Plotting Parameters
# ==========================================
fs = 21
lw_global = 1.0 

plt.switch_backend('pgf')
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": fs,
})

# ==========================================
# 2. Plotting Functions
# ==========================================
def _style_and_buffer_axes(ax, ax_twin, buffer_ratio=0.10):
    """Helper function to apply labels, ticks, grid, and y-axis buffers."""
    ax.set_ylabel(r'$\langle \sigma_{i}^{z} \rangle$', fontsize=fs)
    ax_twin.set_ylabel(r'$\left\langle \frac{1-\sigma_i^z \sigma_{i+1}^z}{2} \right\rangle$', fontsize=fs)
    
    ax.tick_params(axis='both', labelsize=fs-4)
    ax_twin.tick_params(axis='y', labelsize=fs-4)
    ax.grid(alpha=0.25)
    
    # Apply buffer to both axes
    for a in [ax, ax_twin]:
        ymin, ymax = a.get_ylim()
        a.set_ylim(ymin, ymax + buffer_ratio * (ymax - ymin))

def plot_six_panel_qsim_mps_grouped(lam, tq_targets, h, qsim_starts, tai_targets, hz_mps, sigma_dis_mps, window=100, figsize=(10, 20), lw=1.5):
    fig = plt.figure(figsize=figsize)
    gs_main = gridspec.GridSpec(3, 1, figure=fig, hspace=0.125) 
    
    panel_labels = ['(a)', '(b)', '(c)']
    bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=1.0, lw=1.0)
    ticks = np.arange(0, window + 1, 20)

    for idx, (label, tq, qsim_start, tai_mps) in enumerate(zip(panel_labels, tq_targets, qsim_starts, tai_targets)):
        gs_pair = gs_main[idx].subgridspec(2, 1, hspace=0.05) 
        ax_qsim = fig.add_subplot(gs_pair[0])
        ax_mps = fig.add_subplot(gs_pair[1])
        
        # ------------------------------------------
        # TOP PANEL: Q Sim Data
        # ------------------------------------------
        if tq is not None:
            ax_qsim_twin = ax_qsim.twinx()
            
            l1, = ax_qsim.plot(x_local[tq], mag_slice[tq], lw=lw, color='C0', ls='--', label=r'$\langle \sigma_i^z \rangle$')
            l2, = ax_qsim_twin.plot(bond_x_local[tq], kink_slice[tq], lw=lw, color='C1', label=r'$\left\langle \frac{1-\sigma_i^z \sigma_{i+1}^z}{2} \right\rangle$')

            _style_and_buffer_axes(ax_qsim, ax_qsim_twin)

            ax_qsim.text(0.01, 0.9, rf'$\mathrm{{Q~Sim}}$', transform=ax_qsim.transAxes, fontsize=fs-2)
            ax_qsim.set_title(rf'{label} $\tau_Q = {tq:.3}$', fontsize=fs, pad=12, loc='center')

        # ------------------------------------------
        # BOTTOM PANEL: Noisy MPS Data
        # ------------------------------------------
        if tai_mps is not None and tai_mps in final_Mz_spatial and tai_mps in final_Ezz_spatial:
            ax_mps_twin = ax_mps.twinx()
            
            mz_mps = -final_Mz_spatial[tai_mps]
            mzz_mps = 0.5 - 0.5 * np.array(final_Ezz_spatial[tai_mps])
            
            ax_mps.plot(np.arange(len(mz_mps)), mz_mps, color='C0', ls='--', lw=lw)
            ax_mps_twin.plot(np.arange(len(mzz_mps)) + 0.5, mzz_mps, color='C1', ls='-', lw=lw)

            _style_and_buffer_axes(ax_mps, ax_mps_twin)

            ax_mps.text(0.01, 0.9, rf'$\mathrm{{disordered~MPS}}~(\sigma={sigma_dis_mps})$', transform=ax_mps.transAxes, fontsize=fs-2)
            
            if idx == 2: # Add legend and parameters box to the final MPS panel
                ax_mps.legend([l1, l2], [l1.get_label(), l2.get_label()], fontsize=fs-4, loc='center left', frameon=False, ncol=2)
                ax_mps.text(0.035, 0.175, rf'$\lambda={lam:.3f}$, $h = {hz_mps:.3f}$', transform=ax_mps.transAxes, fontsize=fs, bbox=bbox_props)
        else:
            ax_mps.text(0.5, 0.5, 'noisy MPS data not yet available', ha='center', va='center', transform=ax_mps.transAxes, fontsize=fs)
            ax_mps.set_yticks([])

        # ------------------------------------------
        # Enforce limits & hide inner x-labels
        # ------------------------------------------
        for ax in [ax_qsim, ax_mps]:
            ax.set_xlim(0, window)
            ax.set_xticks(ticks)
        
        ax_qsim.tick_params(labelbottom=False)
        if idx < 2:
            ax_mps.tick_params(labelbottom=False)
        
        if idx == 2:
            ax_mps.set_xlabel(r'Site index $i$', fontsize=fs)

    plt.savefig('Fig_4.pdf', dpi=300, bbox_inches='tight')
    plt.close(fig) 

# ==========================================
# 3. Main Execution 
# ==========================================
print("Starting grouped 6-panel plot generation...")

# Use list comprehensions for cleaner data extraction
tai_indices = [0, 1, 2]
tai_targets = [valid_ta_list[i] if i < len(valid_ta_list) else None for i in tai_indices]
qsim_starts_list = [1500, 1800, 1000] 

plot_six_panel_qsim_mps_grouped(
    lam=0.554134,
    tq_targets=np.array([1.559576, 6.479758, 26.922237]),
    h=0.081113,
    qsim_starts=qsim_starts_list, 
    tai_targets=tai_targets,
    hz_mps=0.081113,             
    sigma_dis_mps=0.1,     
    window=100,                  
    figsize=(10, 20),            
    lw=2 * lw_global             
)

print("Generated grouped 6-panel plot with data buffers.")

Starting grouped 6-panel plot generation...


/home/fbayocbocjr/miniconda3/envs/xy/lib/python3.12/site-packages/matplotlib/cbook.py:1709: ComplexWarning: Casting complex values to real discards the imaginary part
  return math.isfinite(val)
/home/fbayocbocjr/miniconda3/envs/xy/lib/python3.12/site-packages/matplotlib/cbook.py:1345: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)


Generated grouped 6-panel plot with data buffers.
